In [ ]:
import random
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/all-MiniLM-L6-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 256 if device == "mps" else 64
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})

In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df["sentence1"] = df["sentence1"].astype(str)
df["sentence2"] = df["sentence2"].astype(str)
df["label"] = df["label"].astype(np.float32)

all_sentences = pd.concat([df["sentence1"], df["sentence2"]], ignore_index=True)
unique_sentences = pd.Index(pd.unique(all_sentences))

num_pairs = len(df)
total_sentence_occurrences = len(all_sentences)
num_unique_sentences = len(unique_sentences)
duplicate_occurrences = total_sentence_occurrences - num_unique_sentences
cache_hit_rate = duplicate_occurrences / total_sentence_occurrences if total_sentence_occurrences else 0.0
dedup_reduction_rate = 1.0 - (num_unique_sentences / total_sentence_occurrences) if total_sentence_occurrences else 0.0

print({
    "num_examples": num_pairs,
    "total_sentence_occurrences": total_sentence_occurrences,
    "unique_sentences": num_unique_sentences,
    "duplicate_occurrences": duplicate_occurrences,
    "cache_hit_rate": round(cache_hit_rate, 6),
    "dedup_reduction_rate": round(dedup_reduction_rate, 6),
})
print(df.head())

In [ ]:
model = SentenceTransformer(model_name, device=device)
model.eval()
print(model_name)

In [ ]:
encoding_start_time = time.time()

unique_embeddings = model.encode(
    unique_sentences.tolist(),
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

sentence_to_idx = {sentence: idx for idx, sentence in enumerate(unique_sentences.tolist())}
idx1 = df["sentence1"].map(sentence_to_idx).to_numpy(dtype=np.int32)
idx2 = df["sentence2"].map(sentence_to_idx).to_numpy(dtype=np.int32)

emb1 = unique_embeddings[idx1]
emb2 = unique_embeddings[idx2]

cosine_similarity = np.sum(emb1 * emb2, axis=1)
predicted_score_0_5 = 2.5 * (cosine_similarity + 1.0)
labels = df["label"].to_numpy(dtype=np.float32)

encoding_runtime_seconds = time.time() - encoding_start_time

print({
    "embedding_matrix_shape": tuple(unique_embeddings.shape),
    "reconstructed_pairs": len(cosine_similarity),
    "encoding_runtime_seconds": round(encoding_runtime_seconds, 2),
})

In [ ]:
spearman_corr = spearmanr(predicted_score_0_5, labels).statistic
pearson_corr = pearsonr(predicted_score_0_5, labels).statistic

results_df = df.copy()
results_df["sentence1_cache_idx"] = idx1
results_df["sentence2_cache_idx"] = idx2
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["error"] = results_df["predicted_score_0_5"] - results_df["label"]
results_df["absolute_error"] = np.abs(results_df["error"])

score_mean = float(np.mean(predicted_score_0_5))
score_std = float(np.std(predicted_score_0_5))
label_mean = float(np.mean(labels))
label_std = float(np.std(labels))

print(results_df[[
    "sentence1", "sentence2", "label", "cosine_similarity", "predicted_score_0_5", "error", "absolute_error"
]].head(10))

In [ ]:
top_k = 5

most_overestimated = results_df.nlargest(top_k, "error")[[
    "sentence1", "sentence2", "label", "predicted_score_0_5", "cosine_similarity", "error", "absolute_error"
]].reset_index(drop=True)

most_underestimated = results_df.nsmallest(top_k, "error")[[
    "sentence1", "sentence2", "label", "predicted_score_0_5", "cosine_similarity", "error", "absolute_error"
]].reset_index(drop=True)

print("Most over-estimated pairs:")
print(most_overestimated.to_string(index=False))
print()
print("Most under-estimated pairs:")
print(most_underestimated.to_string(index=False))

In [ ]:
runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples: {num_pairs}")
print(f"total_sentence_occurrences: {total_sentence_occurrences}")
print(f"unique_sentences_encoded_once: {num_unique_sentences}")
print(f"duplicate_occurrences_served_from_cache: {duplicate_occurrences}")
print(f"cache_hit_rate: {cache_hit_rate:.6f}")
print(f"dedup_reduction_rate: {dedup_reduction_rate:.6f}")
print(f"spearman_correlation: {spearman_corr:.6f}")
print(f"pearson_correlation: {pearson_corr:.6f}")
print(f"predicted_score_mean: {score_mean:.6f}")
print(f"predicted_score_std: {score_std:.6f}")
print(f"label_mean: {label_mean:.6f}")
print(f"label_std: {label_std:.6f}")
print(f"encoding_runtime_seconds: {encoding_runtime_seconds:.2f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")